<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day10-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 10, Segment 2 discussion — does kernel size matter, on the real localization data?

The main notebook's 1D-CNN used a fixed kernel size of 7. Here: re-fetch the exact same subcellular-localization dataset/split from Days 8-10, and compare three kernel sizes (3, 7, 15) on real receptive field and real measured validation accuracy, rather than assuming 7 was the right choice.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import re
import requests

torch.manual_seed(0)
np.random.seed(0)

UNIPROT_CLASSES = {
    "Cytoplasm": "SL-0086",
    "Nucleus": "SL-0191",
    "Mitochondrion": "SL-0173",
    "Secreted": "SL-0243",
    "Cell membrane": "SL-0039",
}
CLASS_NAMES = list(UNIPROT_CLASSES.keys())

def fetch_uniprot_class(location_id, per_class=200):
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"(cc_scl_term:{location_id}) AND (reviewed:true) AND (organism_id:9606) "
                 f"AND (length:[50 TO 500])",
        "fields": "accession,sequence,cc_subcellular_location",
        "format": "json",
        "size": per_class,
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    seqs = []
    for entry in response.json().get("results", []):
        seq = entry.get("sequence", {}).get("value")
        if seq:
            seqs.append(seq)
    return seqs

sequences_by_class = {name: fetch_uniprot_class(loc_id) for name, loc_id in UNIPROT_CLASSES.items()}
for name, seqs in sequences_by_class.items():
    print(f"{name}: {len(seqs)} sequences fetched")

Cytoplasm: 200 sequences fetched
Nucleus: 200 sequences fetched
Mitochondrion: 200 sequences fetched
Secreted: 200 sequences fetched
Cell membrane: 200 sequences fetched


In [2]:
N_PER_CLASS = min(140, min(len(v) for v in sequences_by_class.values()))
print("using", N_PER_CLASS, "sequences per class (matches Days 8-10's 140/class)")

rng = np.random.RandomState(0)
balanced_seqs, balanced_labels = [], []
for class_idx, class_name in enumerate(CLASS_NAMES):
    chosen = rng.choice(len(sequences_by_class[class_name]), size=N_PER_CLASS, replace=False)
    for i in chosen:
        balanced_seqs.append(sequences_by_class[class_name][i])
        balanced_labels.append(class_idx)

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
SEQ_LEN = 150

def one_hot_encode(sequence, length=SEQ_LEN):
    encoded = np.zeros((length, len(AMINO_ACIDS)), dtype=np.float32)
    for position, residue in enumerate(sequence[:length]):
        idx = AA_TO_INDEX.get(residue)
        if idx is not None:
            encoded[position, idx] = 1.0
    return encoded

X = np.stack([one_hot_encode(s) for s in balanced_seqs]).transpose(0, 2, 1)  # (N, 20, 150)
y = np.array(balanced_labels)
print("X shape:", X.shape, " y shape:", y.shape)

using 140 sequences per class (matches Days 8-10's 140/class)
X shape: (700, 20, 150)  y shape: (700,)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=0)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=0)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)
print(f"train {len(X_train)}, val {len(X_val)}, test {len(X_test)}")

train 490, val 105, test 105


## Three kernel sizes, same architecture otherwise

Each filter's **receptive field** after two stacked convolutional layers is
$k + (k - 1) = 2k - 1$ residues (each layer adds $k-1$ to how far a single output position can "see" back along the sequence). Train the same two-conv-layer architecture with kernel size 3, 7, and 15, and compare real measured validation accuracy and parameter count.

In [4]:
class KernelSizeCNN(nn.Module):
    def __init__(self, kernel_size, n_classes=5, n_channels=20):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(n_channels, 32, kernel_size=kernel_size, padding=pad)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=kernel_size, padding=pad)
        self.pool = nn.MaxPool1d(2)
        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = torch.adaptive_avg_pool1d(x, 1).squeeze(-1)
        return self.classifier(x)


def train_and_eval(kernel_size, epochs=60, lr=1e-3):
    torch.manual_seed(0)
    model = KernelSizeCNN(kernel_size)
    n_params = sum(p.numel() for p in model.parameters())
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best_val_acc = 0.0
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_train_t), y_train_t)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_acc = (model(X_val_t).argmax(1) == y_val_t).float().mean().item()
        best_val_acc = max(best_val_acc, val_acc)
    receptive_field = 2 * kernel_size - 1
    return n_params, best_val_acc, receptive_field


print(f"{'kernel':>6} {'receptive field':>16} {'params':>10} {'best val acc':>13}")
results = {}
for k in (3, 7, 15):
    n_params, best_val_acc, rf = train_and_eval(k)
    results[k] = (n_params, best_val_acc, rf)
    print(f"{k:>6} {rf:>16} {n_params:>10,} {best_val_acc:>13.3f}")

kernel  receptive field     params  best val acc


     3                5      8,485         0.505


     7               13     19,237         0.543


    15               29     40,741         0.533


## Discuss

1. Before running the cell above, guess: will the largest kernel (15) win? A bigger kernel sees more context per layer, but also has more weights to learn from only 490 training examples.
2. Does the real result match Day 10's main finding (weight sharing buys parameter efficiency, not necessarily a clear accuracy win on this small dataset)?

*One group presents.*